# 🧠 Steering Diffusion Language Models via SAE Feature Intervention

**Extending DLM-Scope (ICLR 2026) for Controllable Reasoning**

Self-contained notebook — no external imports needed. Run all cells in order.

**Runtime**: Change to T4 GPU → `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ============================================================
# CELL 1: Install dependencies & check GPU
# ============================================================
!pip install -q transformers accelerate datasets scipy seaborn safetensors tqdm

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# ============================================================
# CELL 2: Top-K Sparse Autoencoder (DLM-Scope architecture)
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import json

class TopKSAE(nn.Module):
    """
    Top-K Sparse Autoencoder from DLM-Scope (ICLR 2026).
    Architecture: pre-encoder bias → linear → ReLU → TopK → linear decode
    """
    def __init__(self, d_model: int, d_dict: int, k: int = 64):
        super().__init__()
        self.d_model = d_model
        self.d_dict = d_dict
        self.k = k

        self.encoder = nn.Linear(d_model, d_dict, bias=True)
        self.decoder = nn.Linear(d_dict, d_model, bias=True)
        self.pre_bias = nn.Parameter(torch.zeros(d_model))

        # Init decoder columns to unit norm
        with torch.no_grad():
            self.decoder.weight.data = F.normalize(self.decoder.weight.data, dim=0)

        self.feature_activation_counts = torch.zeros(d_dict, dtype=torch.long)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        x_centered = x - self.pre_bias
        h_pre = self.encoder(x_centered)
        h_relu = F.relu(h_pre)
        # Top-K: zero out all but top-k activations
        topk_vals, topk_idx = h_relu.topk(self.k, dim=-1)
        h_sparse = torch.zeros_like(h_relu)
        h_sparse.scatter_(-1, topk_idx, topk_vals)
        return h_sparse

    def decode(self, h: torch.Tensor) -> torch.Tensor:
        return self.decoder(h) + self.pre_bias

    def forward(self, x):
        h = self.encode(x)
        x_hat = self.decode(h)
        recon_loss = F.mse_loss(x_hat, x)
        l0 = (h > 0).float().sum(dim=-1).mean()
        # Explained variance
        total_var = (x - x.mean(dim=0)).pow(2).sum()
        resid_var = (x - x_hat).pow(2).sum()
        ev = 1.0 - resid_var / (total_var + 1e-8)
        # Track active features
        active = (h > 0).any(dim=0) if h.dim() == 2 else (h > 0).any(dim=0).any(dim=0)
        self.feature_activation_counts += active.long().cpu()
        return x_hat, h, {'recon_loss': recon_loss, 'l0': l0, 'explained_variance': ev}

    def get_feature_directions(self, feature_indices):
        """Get decoder weight columns for given features. Returns [d_model, n_features]."""
        return self.decoder.weight[:, feature_indices].detach()

    @property
    def num_dead_features(self):
        return int((self.feature_activation_counts == 0).sum())

    def _normalize_decoder(self):
        with torch.no_grad():
            self.decoder.weight.data = F.normalize(self.decoder.weight.data, dim=0)

    def save(self, path):
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        torch.save(self.state_dict(), path / 'sae_weights.pt')
        with open(path / 'sae_config.json', 'w') as f:
            json.dump({'d_model': self.d_model, 'd_dict': self.d_dict, 'k': self.k}, f)

    @classmethod
    def load(cls, path):
        path = Path(path)
        with open(path / 'sae_config.json') as f:
            cfg = json.load(f)
        sae = cls(**cfg)
        sae.load_state_dict(torch.load(path / 'sae_weights.pt', map_location='cpu'))
        return sae

print('✅ TopKSAE defined')

In [ ]:
# ============================================================
# CELL 3: DLM Wrapper — Load DiffuGPT with weight remapping
# ============================================================
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from safetensors.torch import load_file as load_safetensors
from huggingface_hub import hf_hub_download
import logging

logger = logging.getLogger('dlm')
logging.basicConfig(level=logging.INFO)

class DLMWrapper:
    MASK_TOKEN_ID = 50257  # DiffuGPT's mask token (index = original vocab_size)

    def __init__(self, model_name='diffusionfamily/diffugpt-m',
                 base_model_name='gpt2-medium', cache_dir=None):
        self.model_name = model_name
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.cache_dir = cache_dir
        self._activations = {}
        self._hooks = []

        # 1. Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name, cache_dir=cache_dir)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        # 2. Load GPT-2 Medium architecture
        print('Loading GPT-2 Medium architecture...')
        self.model = AutoModelForCausalLM.from_pretrained(
            base_model_name, cache_dir=cache_dir,
            torch_dtype=torch.float16 if self.device.type == 'cuda' else torch.float32,
        )

        # 3. Resize to 50258 (DiffuGPT added 1 mask token)
        self.model.resize_token_embeddings(50258)
        self.mask_token_id = 50257

        # 4. Load & remap DiffuGPT weights
        self._load_diffugpt_weights()

        self.model = self.model.to(self.device)
        self.model.eval()

        self.config = self.model.config
        self.d_model = self.config.n_embd  # 1024
        self.n_layers = self.config.n_layer  # 24
        print(f'✅ DiffuGPT loaded: d_model={self.d_model}, n_layers={self.n_layers}, device={self.device}')

    def _load_diffugpt_weights(self):
        """Download DiffuGPT checkpoint and remap weight names to GPT-2 format."""
        print('Downloading DiffuGPT weights...')
        ckpt_path = hf_hub_download(self.model_name, 'model.safetensors', cache_dir=self.cache_dir)
        raw_state = load_safetensors(ckpt_path)

        # Remap: denoise_model.* -> transformer.*, embed_tokens -> wte + lm_head
        mapped = {}
        for key, val in raw_state.items():
            if key.startswith('denoise_model.'):
                mapped['transformer.' + key[len('denoise_model.'):]] = val
            elif key == 'embed_tokens.weight':
                mapped['transformer.wte.weight'] = val
                mapped['lm_head.weight'] = val.clone()
            else:
                mapped[key] = val

        result = self.model.load_state_dict(mapped, strict=False)
        print(f'  Mapped {len(mapped)} params, {len(result.missing_keys)} missing, {len(result.unexpected_keys)} unexpected')
        if result.missing_keys:
            print(f'  Missing: {result.missing_keys[:5]}...')

    def _get_layer_module(self, idx):
        return self.model.transformer.h[idx]

    def register_activation_hooks(self, layers):
        self.clear_hooks()
        for idx in layers:
            layer = self._get_layer_module(idx)
            def make_hook(i):
                def fn(mod, inp, out):
                    self._activations[i] = (out[0] if isinstance(out, tuple) else out).detach().cpu()
                return fn
            self._hooks.append(layer.register_forward_hook(make_hook(idx)))

    def clear_hooks(self):
        for h in self._hooks: h.remove()
        self._hooks, self._activations = [], {}

    def get_activations(self):
        return dict(self._activations)

    @torch.no_grad()
    def forward_pass(self, input_ids, attention_mask=None):
        input_ids = input_ids.to(self.device)
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        return self.model(input_ids=input_ids, attention_mask=attention_mask.to(self.device)).logits

    @torch.no_grad()
    def denoising_loop(self, input_ids, num_steps=30, prefix_len=0, temperature=1.0,
                       steering_hook=None, steering_layer=None,
                       collect_activations=False, activation_layers=None, activation_timesteps=None):
        B, L = input_ids.shape
        gen_len = L - prefix_len
        current = input_ids.clone().to(self.device)
        mask = torch.zeros(B, L, dtype=torch.bool, device=self.device)
        mask[:, prefix_len:] = True
        current[:, prefix_len:] = self.mask_token_id

        trajectory = []
        all_activations = {} if collect_activations else None

        if collect_activations and activation_layers:
            self.register_activation_hooks(activation_layers)
        if activation_timesteps is None:
            activation_timesteps = list(range(num_steps))

        for step in range(num_steps):
            n_unmask = min(int(gen_len * (step + 1) / num_steps), gen_len)

            # Steering hook
            steer_handle = None
            if steering_hook and steering_layer is not None:
                layer_mod = self._get_layer_module(steering_layer)
                def _steer(mod, inp, out, _hook=steering_hook, _mask=mask):
                    if isinstance(out, tuple):
                        return (_hook(out[0], _mask),) + out[1:]
                    return _hook(out, _mask)
                steer_handle = layer_mod.register_forward_hook(_steer)

            logits = self.forward_pass(current)

            if steer_handle:
                steer_handle.remove()

            if collect_activations and step in activation_timesteps:
                all_activations[step] = {l: a.clone() for l, a in self._activations.items()}

            if temperature > 0:
                probs = F.softmax(logits / temperature, dim=-1)
                predicted = torch.multinomial(probs.view(-1, probs.size(-1)), 1).view(B, L)
            else:
                predicted = logits.argmax(dim=-1)

            confidences = F.softmax(logits, dim=-1).gather(-1, predicted.unsqueeze(-1)).squeeze(-1)

            filled = current.clone()
            filled[mask] = predicted[mask]

            if step < num_steps - 1:
                gen_conf = confidences[:, prefix_len:].clone()
                gen_conf[~mask[:, prefix_len:]] = float('inf')
                _, keep = gen_conf.topk(n_unmask, dim=-1)
                new_mask = torch.ones(B, L, dtype=torch.bool, device=self.device)
                new_mask[:, :prefix_len] = False
                for b in range(B):
                    new_mask[b, prefix_len + keep[b]] = False
                current = filled.clone()
                current[new_mask] = self.mask_token_id
                mask = new_mask
            else:
                current = filled
                mask = torch.zeros_like(mask)

            trajectory.append(current.cpu().clone())

        if collect_activations:
            self.clear_hooks()

        return {'output_ids': current.cpu(), 'trajectory': trajectory, 'activations': all_activations}

    def generate(self, prompt, max_new_tokens=128, num_steps=30, temperature=1.0,
                 steering_hook=None, steering_layer=None):
        prompt_ids = self.tokenizer.encode(prompt, return_tensors='pt')
        prefix_len = prompt_ids.shape[1]
        gen_ids = torch.full((1, max_new_tokens), self.mask_token_id, dtype=torch.long)
        input_ids = torch.cat([prompt_ids, gen_ids], dim=1)
        result = self.denoising_loop(input_ids, num_steps, prefix_len, temperature,
                                     steering_hook, steering_layer)
        return self.tokenizer.decode(result['output_ids'][0], skip_special_tokens=True)

    def get_model_info(self):
        return {'d_model': self.d_model, 'n_layers': self.n_layers,
                'device': str(self.device), 'mask_token_id': self.mask_token_id}

print('✅ DLMWrapper defined')

In [ ]:
# ============================================================
# CELL 4: Load DiffuGPT & Quick Test
# ============================================================
import os
SAVE_DIR = '/content/dlm_results'
os.makedirs(SAVE_DIR, exist_ok=True)

dlm = DLMWrapper(
    model_name='diffusionfamily/diffugpt-m',
    base_model_name='gpt2-medium',
    cache_dir=os.path.join(SAVE_DIR, 'cache'),
)

# Quick test
print('\n🧪 Generation test:')
output = dlm.generate('The capital of France is', max_new_tokens=32, num_steps=20, temperature=0.7)
print(f'  Output: {output}')

# Activation test
print('\n🧪 Activation test:')
dlm.register_activation_hooks([4, 12, 20])
_ = dlm.forward_pass(dlm.tokenizer.encode('hello world', return_tensors='pt'))
for l, a in dlm.get_activations().items():
    print(f'  Layer {l}: {a.shape}')
dlm.clear_hooks()
print('\n✅ Phase 1 complete')

In [ ]:
# ============================================================
# CELL 5: GSM8K Data Loading
# ============================================================
from datasets import load_dataset
import re

class GSM8KLoader:
    COT_TEMPLATE = 'Solve this math problem step by step.\nQuestion: {q}\nSolution:'
    DIRECT_TEMPLATE = 'What is the numerical answer?\nQuestion: {q}\nAnswer:'

    def __init__(self, n_problems=200, split='test'):
        print(f'Loading GSM8K ({split}, {n_problems} problems)...')
        ds = load_dataset('openai/gsm8k', 'main', split=split)
        self.problems = list(ds.select(range(min(n_problems, len(ds)))))
        print(f'  Loaded {len(self.problems)} problems')

    def get_cot_prompts(self, indices=None):
        probs = [self.problems[i] for i in indices] if indices else self.problems
        return [{'prompt': self.COT_TEMPLATE.format(q=p['question']),
                 'answer': self._extract_answer(p['answer']),
                 'question': p['question']} for p in probs]

    def get_direct_prompts(self, indices=None):
        probs = [self.problems[i] for i in indices] if indices else self.problems
        return [{'prompt': self.DIRECT_TEMPLATE.format(q=p['question']),
                 'answer': self._extract_answer(p['answer']),
                 'question': p['question']} for p in probs]

    @staticmethod
    def _extract_answer(answer_text):
        match = re.search(r'####\s*([\d,\.\-]+)', answer_text)
        if match:
            return match.group(1).replace(',', '')
        nums = re.findall(r'-?[\d,]+\.?\d*', answer_text)
        return nums[-1].replace(',', '') if nums else '0'

gsm8k = GSM8KLoader(n_problems=200)

# Split: first 100 for discovery, last 100 for evaluation
discovery_idx = list(range(100))
eval_idx = list(range(100, 200))

print(f'\nSample CoT prompt:\n{gsm8k.get_cot_prompts([0])[0]["prompt"][:200]}')
print(f'\nSample Direct prompt:\n{gsm8k.get_direct_prompts([0])[0]["prompt"][:200]}')
print('\n✅ GSM8K loaded')

In [ ]:
# ============================================================
# CELL 6: Collect Activations (Phase 2)
# ============================================================
from tqdm import tqdm
import gc

TARGET_LAYERS = [4, 8, 12, 16, 20]
N_DISCOVERY = 50  # Use 50 problems per condition for speed
MAX_TOKENS = 64  # Generate 64 tokens per prompt
DENOISING_STEPS = 20

def collect_activations(dlm, prompts, layers, n_problems, max_tokens=64, steps=20):
    """Collect residual stream activations from multiple prompts."""
    all_acts = {l: [] for l in layers}

    for i, p in enumerate(tqdm(prompts[:n_problems], desc='Collecting')):
        prompt_ids = dlm.tokenizer.encode(p['prompt'], return_tensors='pt',
                                          max_length=128, truncation=True)
        prefix_len = prompt_ids.shape[1]
        gen_ids = torch.full((1, max_tokens), dlm.mask_token_id, dtype=torch.long)
        input_ids = torch.cat([prompt_ids, gen_ids], dim=1)

        result = dlm.denoising_loop(
            input_ids, num_steps=steps, prefix_len=prefix_len,
            temperature=0.8, collect_activations=True,
            activation_layers=layers,
            activation_timesteps=[steps // 2],  # Middle timestep
        )

        mid_step = steps // 2
        if result['activations'] and mid_step in result['activations']:
            for l in layers:
                if l in result['activations'][mid_step]:
                    act = result['activations'][mid_step][l].float()
                    # Take mean over sequence positions
                    all_acts[l].append(act.mean(dim=1))  # [1, d_model]

        if (i + 1) % 10 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    # Stack into tensors
    return {l: torch.cat(acts, dim=0) for l, acts in all_acts.items() if acts}

print('📊 Collecting CoT activations...')
cot_prompts = gsm8k.get_cot_prompts(discovery_idx)
cot_acts = collect_activations(dlm, cot_prompts, TARGET_LAYERS, N_DISCOVERY,
                                max_tokens=MAX_TOKENS, steps=DENOISING_STEPS)
print(f'  CoT: {{", ".join(f"L{l}: {a.shape}" for l, a in cot_acts.items())}}')

print('\n📊 Collecting Direct activations...')
direct_prompts = gsm8k.get_direct_prompts(discovery_idx)
direct_acts = collect_activations(dlm, direct_prompts, TARGET_LAYERS, N_DISCOVERY,
                                   max_tokens=MAX_TOKENS, steps=DENOISING_STEPS)
print(f'  Direct: {{", ".join(f"L{l}: {a.shape}" for l, a in direct_acts.items())}}')

# Save
torch.save({'cot': cot_acts, 'direct': direct_acts}, os.path.join(SAVE_DIR, 'activations.pt'))
print('\n✅ Phase 2 complete: activations saved')

In [ ]:
# ============================================================
# CELL 7: Train SAE (Phase 3)
# ============================================================
from torch.utils.data import DataLoader, TensorDataset

PRIMARY_LAYER = 16  # Deep layer — best for steering per DLM-Scope

# Combine CoT + Direct activations for training
train_data = torch.cat([cot_acts[PRIMARY_LAYER], direct_acts[PRIMARY_LAYER]], dim=0)
print(f'Training SAE on layer {PRIMARY_LAYER}: {train_data.shape}')

d_model = train_data.shape[1]
d_dict = 4 * d_model  # 4096 features
K = 64

sae = TopKSAE(d_model=d_model, d_dict=d_dict, k=K)
sae = sae.to(dlm.device)

optimizer = torch.optim.Adam(sae.parameters(), lr=3e-4)
loader = DataLoader(TensorDataset(train_data), batch_size=32, shuffle=True)

N_EPOCHS = 10
print(f'Training: {N_EPOCHS} epochs, d_dict={d_dict}, k={K}')

for epoch in range(N_EPOCHS):
    total_loss, total_ev, n = 0, 0, 0
    for (batch,) in loader:
        batch = batch.to(dlm.device)
        x_hat, h, metrics = sae(batch)
        loss = metrics['recon_loss']
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sae.parameters(), 1.0)
        optimizer.step()
        sae._normalize_decoder()
        total_loss += loss.item()
        total_ev += metrics['explained_variance'].item()
        n += 1
    print(f'  Epoch {epoch+1}/{N_EPOCHS}: loss={total_loss/n:.4f}, EV={total_ev/n:.3f}, dead={sae.num_dead_features}/{d_dict}')

sae.eval()
sae.save(os.path.join(SAVE_DIR, 'sae_layer16'))
print(f'\n✅ Phase 3 complete: SAE trained (EV={total_ev/n:.3f})')

In [ ]:
# ============================================================
# CELL 8: Contrastive Feature Discovery (Phase 4) — NOVEL CONTRIBUTION
# ============================================================
import numpy as np
from scipy import stats

def contrastive_feature_analysis(sae, cot_data, direct_data, alpha=0.05, min_d=0.2):
    """Identify features differentially active during CoT vs direct answering."""
    sae.eval()
    device = next(sae.parameters()).device

    # Encode both conditions through SAE
    with torch.no_grad():
        cot_feats = sae.encode(cot_data.to(device)).cpu().numpy()
        direct_feats = sae.encode(direct_data.to(device)).cpu().numpy()

    n_features = cot_feats.shape[1]
    results = {'diff': [], 'p_val': [], 'cohens_d': [], 'cot_mean': [], 'direct_mean': []}

    for f in range(n_features):
        c, d = cot_feats[:, f], direct_feats[:, f]
        cm, dm = c.mean(), d.mean()
        cs, ds = c.std() + 1e-8, d.std() + 1e-8

        diff = cm - dm
        if cs > 1e-7 or ds > 1e-7:
            t, p = stats.ttest_ind(c, d, equal_var=False)
            p_one = p / 2 if t > 0 else 1 - p / 2
        else:
            p_one = 1.0

        pooled = np.sqrt((cs**2 + ds**2) / 2)
        cd = diff / (pooled + 1e-8)

        results['diff'].append(diff)
        results['p_val'].append(p_one)
        results['cohens_d'].append(cd)
        results['cot_mean'].append(cm)
        results['direct_mean'].append(dm)

    results = {k: np.array(v) for k, v in results.items()}

    # Bonferroni correction
    sig = results['p_val'] < (alpha / n_features)
    large = np.abs(results['cohens_d']) >= min_d
    reasoning = sig & large & (results['diff'] > 0)
    anti = sig & large & (results['diff'] < 0)

    reasoning_idx = np.where(reasoning)[0]
    reasoning_ranked = reasoning_idx[np.argsort(results['cohens_d'][reasoning_idx])[::-1]]

    anti_idx = np.where(anti)[0]
    anti_ranked = anti_idx[np.argsort(results['cohens_d'][anti_idx])]

    return {
        'reasoning_features': reasoning_ranked.tolist(),
        'anti_reasoning_features': anti_ranked.tolist(),
        'n_significant': int(sig.sum()),
        'n_reasoning': int(reasoning.sum()),
        'n_anti': int(anti.sum()),
        **results,
    }

print('🔬 Running contrastive analysis...')
feature_results = contrastive_feature_analysis(
    sae, cot_acts[PRIMARY_LAYER], direct_acts[PRIMARY_LAYER]
)

reasoning_features = feature_results['reasoning_features'][:50]
print(f'  Found {feature_results["n_reasoning"]} reasoning features')
print(f'  Found {feature_results["n_anti"]} anti-reasoning features')
print(f'  Top 10 reasoning features: {reasoning_features[:10]}')

# If no significant features found, use top features by effect size anyway
if len(reasoning_features) == 0:
    print('  ⚠️ No Bonferroni-significant features. Using top features by effect size.')
    top_by_effect = np.argsort(feature_results['cohens_d'])[::-1][:50]
    reasoning_features = [int(x) for x in top_by_effect if feature_results['diff'][x] > 0][:50]
    print(f'  Using {len(reasoning_features)} features by effect size')

# Save
with open(os.path.join(SAVE_DIR, 'feature_results.json'), 'w') as f:
    json.dump({k: v.tolist() if hasattr(v, 'tolist') else v for k, v in feature_results.items()}, f)

print('\n✅ Phase 4 complete: reasoning features identified')

In [ ]:
# ============================================================
# CELL 9: Steering Experiments (Phase 5)
# ============================================================
import random

class DiffusionSteerer:
    def __init__(self, sae, features, layer, alpha=2.0):
        self.sae = sae
        self.features = features
        self.layer = layer
        self.alpha = alpha
        # Pre-compute steering direction
        dirs = sae.get_feature_directions(features)  # [d_model, n_feat]
        self.direction = dirs.sum(dim=1)  # [d_model]
        self.direction = self.direction / (self.direction.norm() + 1e-8)

    def create_hook(self):
        direction = self.direction
        alpha = self.alpha
        def hook(hidden_states, mask):
            device = hidden_states.device
            v = direction.to(device)
            hidden_states = hidden_states + alpha * v.unsqueeze(0).unsqueeze(0)
            return hidden_states
        return hook

def run_experiment(dlm, prompts, sae=None, features=None, layer=None,
                   alpha=0.0, label='baseline', n=20):
    """Generate with optional steering, return results."""
    results = []
    hook, hook_layer = None, None
    if alpha != 0 and features and sae:
        steerer = DiffusionSteerer(sae, features, layer, alpha)
        hook = steerer.create_hook()
        hook_layer = layer

    for i, p in enumerate(tqdm(prompts[:n], desc=label)):
        text = dlm.generate(p['prompt'], max_new_tokens=128, num_steps=20,
                            temperature=0.7, steering_hook=hook, steering_layer=hook_layer)
        results.append({'prompt': p['prompt'], 'generated': text,
                        'answer': p.get('answer', ''), 'condition': label})
    return results

# Evaluation prompts (separate from discovery set)
eval_cot = gsm8k.get_cot_prompts(eval_idx)
N_EVAL = 20

print('=' * 60)
print('[E1] Baseline (no steering)...')
baseline = run_experiment(dlm, eval_cot, n=N_EVAL, label='baseline')

print('\n[E2] Positive steering (α=2.0)...')
steered_pos = run_experiment(dlm, eval_cot, sae, reasoning_features, PRIMARY_LAYER,
                              alpha=2.0, label='positive_2.0', n=N_EVAL)

print('\n[E3] Negative steering (α=-2.0)...')
steered_neg = run_experiment(dlm, eval_cot, sae, reasoning_features, PRIMARY_LAYER,
                              alpha=-2.0, label='negative_-2.0', n=N_EVAL)

print('\n[E4] Random feature control (α=2.0)...')
random.seed(42)
random_feats = random.sample(range(d_dict), 50)
random_ctrl = run_experiment(dlm, eval_cot, sae, random_feats, PRIMARY_LAYER,
                              alpha=2.0, label='random_control', n=N_EVAL)

print('\n✅ Phase 5 complete: all experiments done')

In [ ]:
# ============================================================
# CELL 10: Evaluation — Accuracy & Reasoning Quality
# ============================================================

def extract_number(text):
    """Extract final numerical answer from generated text."""
    # Look for #### pattern first
    m = re.search(r'####\s*([\d,\.\-]+)', text)
    if m: return m.group(1).replace(',', '')
    # Look for 'answer is X'
    m = re.search(r'answer\s+is\s+([\d,\.\-]+)', text, re.I)
    if m: return m.group(1).replace(',', '')
    # Last number in text
    nums = re.findall(r'-?[\d,]+\.?\d*', text)
    return nums[-1].replace(',', '') if nums else None

def reasoning_score(text):
    """Score reasoning quality: higher = more structured reasoning."""
    score = 0
    # Reasoning markers
    markers = ['step', 'first', 'then', 'next', 'therefore', 'so', 'because',
               'since', 'total', 'finally', 'thus', 'hence', 'calculate']
    text_lower = text.lower()
    score += sum(1 for m in markers if m in text_lower)
    # Math operations
    score += len(re.findall(r'[\+\-\*\/\=]', text)) * 0.5
    # Numbers (more = more computation)
    score += len(re.findall(r'\d+', text)) * 0.3
    # Equations/expressions
    score += len(re.findall(r'\d+\s*[\+\-\*\/]\s*\d+', text)) * 2.0
    # Line breaks / steps
    score += text.count('\n') * 0.5
    return score

def evaluate_condition(results, label):
    correct, total = 0, 0
    scores = []
    for r in results:
        pred = extract_number(r['generated'])
        if pred and r['answer']:
            try:
                if abs(float(pred) - float(r['answer'])) < 0.01:
                    correct += 1
            except ValueError:
                pass
        total += 1
        scores.append(reasoning_score(r['generated']))
    acc = correct / total if total > 0 else 0
    avg_score = np.mean(scores) if scores else 0
    return {'condition': label, 'accuracy': acc, 'correct': correct,
            'total': total, 'reasoning_score': avg_score}

print('=' * 70)
print(f'{"Condition":<25} {"Accuracy":<12} {"Reasoning Score":<18} {"Correct/Total"}')
print('-' * 70)

all_evals = {}
for results, label in [(baseline, 'Baseline'),
                         (steered_pos, 'Positive (α=2.0)'),
                         (steered_neg, 'Negative (α=-2.0)'),
                         (random_ctrl, 'Random Control')]:
    ev = evaluate_condition(results, label)
    all_evals[label] = ev
    print(f'{label:<25} {ev["accuracy"]:>8.1%}     {ev["reasoning_score"]:>12.2f}       {ev["correct"]}/{ev["total"]}')

print('=' * 70)

# Show samples
print('\n📝 Sample outputs:')
for i in range(min(3, len(baseline))):
    print(f'\n--- Problem {i+1} ---')
    print(f'Q: {baseline[i]["prompt"][:100]}...')
    print(f'Baseline:  {baseline[i]["generated"][len(baseline[i]["prompt"]):][: 150]}...')
    print(f'Steered+:  {steered_pos[i]["generated"][len(steered_pos[i]["prompt"]):][: 150]}...')
    print(f'Expected:  {baseline[i]["answer"]}')

In [ ]:
# ============================================================
# CELL 11: Alpha Sweep (Phase 5b)
# ============================================================
alpha_values = [0.5, 1.0, 2.0, 3.0, 5.0]
alpha_evals = []

for alpha in alpha_values:
    print(f'\nAlpha sweep: α={alpha}...')
    res = run_experiment(dlm, eval_cot, sae, reasoning_features, PRIMARY_LAYER,
                          alpha=alpha, label=f'alpha_{alpha}', n=15)
    ev = evaluate_condition(res, f'α={alpha}')
    alpha_evals.append({'alpha': alpha, **ev})
    print(f'  Accuracy: {ev["accuracy"]:.1%}, Reasoning Score: {ev["reasoning_score"]:.2f}')

print('\n✅ Alpha sweep complete')

In [ ]:
# ============================================================
# CELL 12: Publication-Quality Visualizations (Phase 6)
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns

# Set dark theme
plt.style.use('dark_background')
sns.set_palette('viridis')
COLORS = {'pos': '#00d4aa', 'neg': '#ff6b6b', 'rand': '#ffd93d', 'base': '#6c8ebf'}
FIG_DIR = os.path.join(SAVE_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

# ---- Figure 1: Top Reasoning Features (Differential Activation) ----
fig, ax = plt.subplots(figsize=(12, 5))
top_n = min(30, len(reasoning_features))
top_feats = reasoning_features[:top_n]
diffs = [feature_results['diff'][f] for f in top_feats]
effects = [feature_results['cohens_d'][f] for f in top_feats]
colors = [plt.cm.viridis(e / (max(effects) + 1e-8)) for e in effects]
ax.barh(range(top_n), diffs, color=colors)
ax.set_yticks(range(top_n))
ax.set_yticklabels([f'Feature {f}' for f in top_feats], fontsize=8)
ax.set_xlabel('Differential Activation (CoT - Direct)', fontsize=12)
ax.set_title('Top Reasoning-Associated SAE Features', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig1_reasoning_features.png'), dpi=150, bbox_inches='tight')
plt.show()

# ---- Figure 2: Steering Results Bar Chart ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

conditions = ['Baseline', 'Positive (α=2.0)', 'Negative (α=-2.0)', 'Random Control']
bar_colors = [COLORS['base'], COLORS['pos'], COLORS['neg'], COLORS['rand']]

# Accuracy
accs = [all_evals[c]['accuracy'] * 100 for c in conditions]
axes[0].bar(range(4), accs, color=bar_colors, edgecolor='white', linewidth=0.5)
axes[0].set_xticks(range(4))
axes[0].set_xticklabels(['Baseline', 'Positive', 'Negative', 'Random'], fontsize=10)
axes[0].set_ylabel('GSM8K Accuracy (%)', fontsize=12)
axes[0].set_title('Accuracy by Condition', fontsize=13, fontweight='bold')
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Reasoning score
rscores = [all_evals[c]['reasoning_score'] for c in conditions]
axes[1].bar(range(4), rscores, color=bar_colors, edgecolor='white', linewidth=0.5)
axes[1].set_xticks(range(4))
axes[1].set_xticklabels(['Baseline', 'Positive', 'Negative', 'Random'], fontsize=10)
axes[1].set_ylabel('Reasoning Quality Score', fontsize=12)
axes[1].set_title('Reasoning Quality by Condition', fontsize=13, fontweight='bold')
for i, v in enumerate(rscores):
    axes[1].text(i, v + 0.2, f'{v:.1f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig2_steering_results.png'), dpi=150, bbox_inches='tight')
plt.show()

# ---- Figure 3: Accuracy vs Alpha ----
if alpha_evals:
    fig, ax = plt.subplots(figsize=(8, 5))
    alphas = [e['alpha'] for e in alpha_evals]
    accs = [e['accuracy'] * 100 for e in alpha_evals]
    scores = [e['reasoning_score'] for e in alpha_evals]

    ax.plot(alphas, accs, 'o-', color=COLORS['pos'], linewidth=2, markersize=8, label='Accuracy')
    ax.axhline(y=all_evals['Baseline']['accuracy'] * 100, color=COLORS['base'],
               linestyle='--', alpha=0.7, label='Baseline')
    ax.set_xlabel('Steering Strength (α)', fontsize=12)
    ax.set_ylabel('GSM8K Accuracy (%)', fontsize=12)
    ax.set_title('Accuracy vs Steering Strength', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'fig3_alpha_sweep.png'), dpi=150, bbox_inches='tight')
    plt.show()

print(f'\n✅ Figures saved to {FIG_DIR}')

In [ ]:
# ============================================================
# CELL 13: Final Summary & Save Everything
# ============================================================

print('=' * 70)
print('🎉 FINAL RESULTS SUMMARY')
print('=' * 70)
print(f'\nModel: DiffuGPT-Medium (355M params)')
print(f'SAE: Top-{K} SAE, d_dict={d_dict}')
print(f'Target Layer: {PRIMARY_LAYER}')
print(f'Reasoning Features Found: {len(reasoning_features)}')
print(f'\n{"Condition":<25} {"Accuracy":<12} {"Reasoning Score"}')
print('-' * 55)
for label, ev in all_evals.items():
    print(f'{label:<25} {ev["accuracy"]:>8.1%}     {ev["reasoning_score"]:>10.2f}')

if alpha_evals:
    print(f'\nAlpha Sweep:')
    for e in alpha_evals:
        print(f'  α={e["alpha"]:<5}: acc={e["accuracy"]:.1%}, reasoning={e["reasoning_score"]:.2f}')

# Save all results
import json
all_results = {
    'model': 'diffusionfamily/diffugpt-m',
    'sae_config': {'d_model': d_model, 'd_dict': d_dict, 'k': K},
    'target_layer': PRIMARY_LAYER,
    'n_reasoning_features': len(reasoning_features),
    'evaluations': all_evals,
    'alpha_sweep': alpha_evals,
    'reasoning_features': reasoning_features,
}
with open(os.path.join(SAVE_DIR, 'final_results.json'), 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print(f'\n📁 All results saved to {SAVE_DIR}')
print('\n📝 Next steps:')
print('  1. Download figures from the Files panel (left sidebar)')
print('  2. Update README.md with real results')
print('  3. Write technical report')
print('\n🏁 Pipeline complete!')